# Merchant Sales Anomaly Detection — Google Colab Final Analytical Notebook
**Original and New merchant portfolios | Big Data Analytics | 2026**

This notebook is the reproducible **analytical companion to the final V6 GitHub outputs**. It contains **no Streamlit code and no application-development or deployment commands**.

It uses the same merchant-level feature engineering and modelling logic, and it **independently regenerates** the processed outputs from the eight assignment source files. It then checks the regenerated results against the exact V6 GitHub reference CSVs, preserving the published reference files for export only after validation.

**Analytical sequence:** Excel/PivotTable preliminary exploration → captured-status joins → merchant-day/category-day preparation → prior-only historical expectations → Isolation Forest → Logistic Regression / Random Forest / Gradient Boosting → external evaluator record → reconciliation and CSV export.

> The course's Model Evaluator hides ground-truth merchant labels. Its performance scores below are recorded historical results, *not* recomputed from the synthetic source data. Matching predictions and their aggregate totals does not prove byte-level identity with every earlier evaluator submission.

## 0. Colab setup and source files

Download **`Merchant_Colab_V6_Inputs.zip`** alongside this notebook. In Google Colab, run this cell and select the ZIP when prompted. It contains eight course-provided raw CSV inputs and twelve **unmodified V6 GitHub output references**. Raw transaction CSVs are only for private Colab analysis and are **not intended for a public GitHub repository**.

The notebook also runs as a local Jupyter notebook if the ZIP is in the working directory. Files remain under a dedicated working directory, and source/reference data are not overwritten.

| Portfolio | Raw inputs |
|---|---|
| Original | `Transactions.csv`, `merchant.csv`, `business.csv`, `status.csv` |
| New | `Transactions_New.csv`, `merchant_New.csv`, `business_New.csv`, `status_New.csv` |

**Excel provenance:** Initial source exploration used `Merchant_Analysis_Original` (eight worksheets, two PivotTables, saved data-quality checks and a chart). Final numerical outputs come from the Python transformations reproduced here.

In [ ]:
import os, sys, zipfile, json, hashlib, shutil
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

WORKDIR = (Path('/content/merchant_anomaly_v6') if Path('/content').exists()
           else Path('/mnt/data/merchant_colab_v6_test'))
RAW_DIR = WORKDIR/'raw_data'
REFERENCE_DIR = WORKDIR/'github_outputs'
REGENERATED_DIR = WORKDIR/'regenerated'
VERIFIED_DIR = WORKDIR/'verified_github_outputs'
WORKDIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_NAME = 'Merchant_Colab_V6_Inputs.zip'
archive_candidates = [Path.cwd()/ARCHIVE_NAME, Path('/mnt/data')/ARCHIVE_NAME]
archive = next((p for p in archive_candidates if p.is_file()), None)
if archive is None and (Path('/content').exists() or 'google.colab' in sys.modules):
    from google.colab import files
    received = files.upload()
    if ARCHIVE_NAME not in received:
        raise FileNotFoundError(f'Upload the exact file: {ARCHIVE_NAME}')
    archive = Path.cwd()/ARCHIVE_NAME
if archive is None:
    raise FileNotFoundError(f'Place {ARCHIVE_NAME} in your notebook working directory.')
with zipfile.ZipFile(archive) as z:
    invalid = [n for n in z.namelist() if Path(n).is_absolute() or '..' in Path(n).parts]
    assert not invalid, f'Unsafe ZIP members: {invalid}'
    z.extractall(WORKDIR)
assert len(list(RAW_DIR.glob('*.csv'))) == 8, 'Expected eight raw source CSVs.'
print('Source data:', RAW_DIR)
print('GitHub reference outputs:', REFERENCE_DIR)
print('Notebook working area:', WORKDIR)
import sklearn
print('Python:',sys.version.split()[0], '| pandas:',pd.__version__, '| scikit-learn:',sklearn.__version__, '| numpy:',np.__version__)
print('Version used for the local full-data reconciliation: pandas 2.2.3, scikit-learn 1.8.0, NumPy 2.3.5.')

## 1. Data understanding and quality checks

Only `Status = Captured` qualifies as recorded sales. Source status codes are resolved against a lookup table; merchant and business reference lookups use validated many-to-one joins. The two portfolios refer to different merchant identities, despite the same January–March observation interval.

In [ ]:
for label, prefix in [('Original',''),('New','_New')]:
    tx_path = RAW_DIR / ('Transactions.csv' if label=='Original' else 'Transactions_New.csv')
    raw = pd.read_csv(tx_path, dtype={'Status_Code':'string'}, parse_dates=['Date'])
    print(label, 'rows:',f'{len(raw):,}', 'merchants:',raw['Merchant_ID'].nunique(),
          'dates:',raw['Date'].nunique(), 'missing amounts:',int(raw['Amount'].isna().sum()),
          'duplicate transaction IDs:',int(raw['Transaction_ID'].duplicated().sum()))
    del raw

## 2. Reproducible transformation functions (same logic as the final V6 pipeline)

The functions below are reproduced directly from the final pipeline's **analysis section only**, without any dashboard code. Original and New datasets share the historical expectation, robust deviation and ten-feature definitions.

**Reference-schema compatibility:** The published *Original* daily file calls its final boolean flag `High_Confidence_Anomaly_Day` and omits the exploratory `Candidate_Extreme_Day`; the *New* daily file retains `High_Confidence_Day` and `Candidate_Extreme_Day`. The Original business file records **USD differences** in `Business_Deviation`, whereas the New business file stores its **log deviation**. Both variants are retained exactly as published rather than silently normalised.

In [ ]:
import argparse
import sys
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    GradientBoostingClassifier,
    IsolationForest,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
FINAL_CONTAMINATION = 0.10
ROBUST_Z_THRESHOLD = 4.0
MIN_RELATIVE_FACTOR = 1.5

FEATURE_COLUMNS = [
    "Sales_CV",
    "Transaction_CV",
    "Peak_Positive_Deviation",
    "Peak_Negative_Deviation",
    "P95_Absolute_Deviation",
    "Mean_Absolute_Deviation",
    "High_Confidence_Day_Count",
    "Max_Robust_Z",
    "Largest_Daily_Log_Change",
    "Period_Shift",
]

ORIGINAL_FILES = {
    "transactions": "Transactions.csv",
    "merchant": "merchant.csv",
    "business": "business.csv",
    "status": "status.csv",
}

NEW_FILES = {
    "transactions": "Transactions_New.csv",
    "merchant": "merchant_New.csv",
    "business": "business_New.csv",
    "status": "status_New.csv",
}

In [ ]:
def clean_status_code(series: pd.Series) -> pd.Series:
    """Normalise status codes such as 1, '1', '01', '1.0' to two-character strings."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(2)
    )


def locate_files(data_dir: Path, mapping: Dict[str, str]) -> Dict[str, Path]:
    paths = {key: data_dir / filename for key, filename in mapping.items()}
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing required files: " + ", ".join(missing))
    return paths


def read_source_files(paths: Dict[str, Path]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    tx = pd.read_csv(paths["transactions"], parse_dates=["Date"], dtype={"Status_Code": "string"})
    merchant = pd.read_csv(paths["merchant"])
    business = pd.read_csv(paths["business"])
    status = pd.read_csv(paths["status"], dtype={"Status_Code": "string"})

    tx["Status_Code"] = clean_status_code(tx["Status_Code"])
    status["Status_Code"] = clean_status_code(status["Status_Code"])
    return tx, merchant, business, status


def prepare_daily_data(
    transactions: pd.DataFrame,
    merchant: pd.DataFrame,
    business: pd.DataFrame,
    status: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Merge references, filter captured sales and create daily merchant/business data."""
    df = (
        transactions
        .merge(merchant, on="Merchant_ID", how="left", validate="many_to_one")
        .merge(business, on="Business_ID", how="left", validate="many_to_one")
        .merge(status, on="Status_Code", how="left", validate="many_to_one")
    )

    captured = df[df["Status"].eq("Captured")].copy()

    calendar = pd.date_range(transactions["Date"].min(), transactions["Date"].max(), freq="D", name="Date")

    daily_agg = (
        captured.groupby(["Merchant_ID", "Date"], as_index=False)
        .agg(Daily_Sales=("Amount", "sum"), Transaction_Count=("Transaction_ID", "size"))
    )

    merchant_calendar = pd.MultiIndex.from_product(
        [merchant["Merchant_ID"].unique(), calendar], names=["Merchant_ID", "Date"]
    ).to_frame(index=False)

    daily = merchant_calendar.merge(daily_agg, on=["Merchant_ID", "Date"], how="left")
    daily[["Daily_Sales", "Transaction_Count"]] = daily[["Daily_Sales", "Transaction_Count"]].fillna(0)
    daily = daily.merge(merchant[["Merchant_ID", "Merchant"]], on="Merchant_ID", how="left")
    daily = daily.sort_values(["Merchant_ID", "Date"]).reset_index(drop=True)

    business_agg = (
        captured.groupby(["Merchant_ID", "Business_ID", "Date"], as_index=False)
        .agg(Business_Sales=("Amount", "sum"), Business_Transactions=("Transaction_ID", "size"))
    )

    business_calendar = pd.MultiIndex.from_product(
        [merchant["Merchant_ID"].unique(), business["Business_ID"].unique(), calendar],
        names=["Merchant_ID", "Business_ID", "Date"],
    ).to_frame(index=False)

    business_daily = business_calendar.merge(
        business_agg, on=["Merchant_ID", "Business_ID", "Date"], how="left"
    )
    business_daily[["Business_Sales", "Business_Transactions"]] = business_daily[
        ["Business_Sales", "Business_Transactions"]
    ].fillna(0)
    business_daily = (
        business_daily
        .merge(merchant[["Merchant_ID", "Merchant"]], on="Merchant_ID", how="left")
        .merge(business[["Business_ID", "Business"]], on="Business_ID", how="left")
        .sort_values(["Merchant_ID", "Business_ID", "Date"])
        .reset_index(drop=True)
    )

    return df, daily, business_daily

### 2.1 Past-only time-series features

Merchant baselines are constructed from previous observations: a 14-day lagged median (minimum seven prior observations), optionally replaced by a weekday expanding median (minimum four previous same-weekday observations). The historical daily flag combines absolute robust Z ≥4 with at least a 1.5-fold absolute log deviation. Early days with insufficient history are not automatically declared normal.

In [ ]:
def add_time_series_features(daily: pd.DataFrame, business_daily: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    daily = daily.copy()
    daily["Weekday"] = daily["Date"].dt.dayofweek
    g = daily.groupby("Merchant_ID", sort=False)

    daily["Prior_14_Median"] = g["Daily_Sales"].transform(
        lambda s: s.shift(1).rolling(14, min_periods=7).median()
    )
    daily["Weekday_Median"] = daily.groupby(["Merchant_ID", "Weekday"])["Daily_Sales"].transform(
        lambda s: s.shift(1).expanding(min_periods=4).median()
    )
    daily["Expected_Sales"] = daily["Weekday_Median"].fillna(daily["Prior_14_Median"])
    daily["Log_Deviation"] = np.log((daily["Daily_Sales"] + 1) / (daily["Expected_Sales"] + 1))
    daily["Abs_Log_Deviation"] = daily["Log_Deviation"].abs()
    daily["Candidate_Extreme_Day"] = daily["Abs_Log_Deviation"].ge(np.log(MIN_RELATIVE_FACTOR))
    daily["Daily_Log_Change"] = daily.groupby("Merchant_ID")["Daily_Sales"].transform(
        lambda s: np.log1p(s).diff().abs()
    )

    parts = []
    for _, grp in daily.groupby("Merchant_ID", sort=False):
        grp = grp.copy()
        shifted = grp["Log_Deviation"].shift(1)
        grp["Residual_Median"] = shifted.rolling(28, min_periods=14).median()
        grp["Residual_MAD"] = shifted.rolling(28, min_periods=14).apply(
            lambda x: np.median(np.abs(x - np.median(x))), raw=True
        )
        grp["Robust_Z"] = 0.6745 * (grp["Log_Deviation"] - grp["Residual_Median"]) / grp[
            "Residual_MAD"
        ].replace(0, np.nan)
        parts.append(grp)
    daily = pd.concat(parts, ignore_index=True).sort_values(["Merchant_ID", "Date"]).reset_index(drop=True)
    daily["High_Confidence_Day"] = (
        daily["Robust_Z"].abs().ge(ROBUST_Z_THRESHOLD)
        & daily["Abs_Log_Deviation"].ge(np.log(MIN_RELATIVE_FACTOR))
    )

    business_daily = business_daily.copy()
    business_daily["Weekday"] = business_daily["Date"].dt.dayofweek
    business_daily["Prior_14_Median"] = business_daily.groupby(["Merchant_ID", "Business_ID"])[
        "Business_Sales"
    ].transform(lambda s: s.shift(1).rolling(14, min_periods=7).median())
    business_daily["Weekday_Median"] = business_daily.groupby(["Merchant_ID", "Business_ID", "Weekday"])[
        "Business_Sales"
    ].transform(lambda s: s.shift(1).expanding(min_periods=4).median())
    business_daily["Expected_Business_Sales"] = business_daily["Weekday_Median"].fillna(
        business_daily["Prior_14_Median"]
    )
    business_daily["Business_Deviation"] = np.log(
        (business_daily["Business_Sales"] + 1) / (business_daily["Expected_Business_Sales"] + 1)
    )
    business_daily["Abs_Business_Deviation"] = business_daily["Business_Deviation"].abs()
    return daily, business_daily

### 2.2 Ten merchant-level modelling features

The feature matrix is fitted on 105 Original merchants, then applied **without feature redesign** to 105 New merchants. Missing-feature imputation uses medians computed from the Original training population.

In [ ]:
def build_merchant_features(daily: pd.DataFrame) -> pd.DataFrame:
    merchant_features = daily.groupby("Merchant_ID").agg(
        Mean_Sales=("Daily_Sales", "mean"),
        SD_Sales=("Daily_Sales", "std"),
        Mean_Transactions=("Transaction_Count", "mean"),
        SD_Transactions=("Transaction_Count", "std"),
        Peak_Positive_Deviation=("Log_Deviation", lambda x: x.clip(lower=0).max()),
        Peak_Negative_Deviation=("Log_Deviation", lambda x: (-x.clip(upper=0)).max()),
        P95_Absolute_Deviation=("Abs_Log_Deviation", lambda x: x.quantile(0.95)),
        Mean_Absolute_Deviation=("Abs_Log_Deviation", "mean"),
        High_Confidence_Day_Count=("High_Confidence_Day", "sum"),
        Max_Robust_Z=("Robust_Z", lambda x: x.abs().max()),
    )

    merchant_features["Sales_CV"] = merchant_features["SD_Sales"] / (merchant_features["Mean_Sales"] + 1)
    merchant_features["Transaction_CV"] = merchant_features["SD_Transactions"] / (
        merchant_features["Mean_Transactions"] + 1
    )
    merchant_features["Largest_Daily_Log_Change"] = daily.groupby("Merchant_ID")["Daily_Log_Change"].max()

    ordered = daily.sort_values(["Merchant_ID", "Date"])
    first_30 = ordered.groupby("Merchant_ID").head(30).groupby("Merchant_ID")["Daily_Sales"].mean()
    last_30 = ordered.groupby("Merchant_ID").tail(30).groupby("Merchant_ID")["Daily_Sales"].mean()
    merchant_features["Period_Shift"] = np.log((last_30 + 1) / (first_30 + 1))
    merchant_features["Max_Robust_Z"] = merchant_features["Max_Robust_Z"].clip(upper=20)
    return merchant_features


def prepare_X(features: pd.DataFrame, medians: Optional[pd.Series] = None) -> Tuple[pd.DataFrame, pd.Series]:
    X = features[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).copy()
    if medians is None:
        medians = X.median()
    X = X.fillna(medians).fillna(0)
    return X, medians


def save_submission(
    merchant_lookup: pd.DataFrame,
    feature_index: Iterable[int],
    predictions: np.ndarray,
    filename: Path,
) -> pd.DataFrame:
    pred_series = pd.Series(predictions, index=pd.Index(feature_index, name="Merchant_ID"), name="Prediction")
    output = merchant_lookup[["Merchant_ID", "Merchant"]].copy()
    output["Prediction"] = output["Merchant_ID"].map(pred_series).astype(int)
    output = output[["Merchant", "Prediction"]].sort_values("Merchant").reset_index(drop=True)

    assert output["Merchant"].notna().all()
    assert output["Merchant"].nunique() == len(output)
    assert set(output["Prediction"].unique()).issubset({0, 1})

    filename.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(filename, index=False)
    print(f"Saved {filename} | rows={len(output)} | anomalies={int(output['Prediction'].sum())}")
    return output

## 3. Unsupervised model — Original portfolio

**Isolation Forest:** 500 estimators, contamination 0.10 and random state 42. A prediction of `1` denotes an anomalous merchant; `0` denotes Normal. Contamination governs an outlier threshold, not the true prevalence of wrongdoing. The V6 merchant-risk score uses **negative Isolation Forest `decision_function`**, so higher recorded values indicate a more anomalous profile.

The subsequent supervised learners are trained on the original Isolation Forest **pseudo-labels**, not human-labelled fraud outcomes.

In [ ]:
def train_original(data_dir: Path, output_dir: Path):
    paths = locate_files(data_dir, ORIGINAL_FILES)
    transactions, merchant, business, status = read_source_files(paths)
    merged, daily, business_daily = prepare_daily_data(transactions, merchant, business, status)
    daily, business_daily = add_time_series_features(daily, business_daily)
    merchant_features = build_merchant_features(daily)
    X, medians = prepare_X(merchant_features)

    iso_model = IsolationForest(
        n_estimators=500,
        contamination=FINAL_CONTAMINATION,
        random_state=RANDOM_STATE,
    )
    iso_pred = (iso_model.fit_predict(X) == -1).astype(int)
    iso_score = -iso_model.decision_function(X)

    merchant_risk = merchant_features.assign(Prediction=iso_pred, Anomaly_Score=iso_score).reset_index()
    merchant_risk = merchant_risk.merge(merchant[["Merchant_ID", "Merchant"]], on="Merchant_ID", how="left")
    merchant_risk = merchant_risk.sort_values("Anomaly_Score", ascending=False).reset_index(drop=True)
    merchant_risk["Risk_Rank"] = np.arange(1, len(merchant_risk) + 1)
    merchant_risk["Risk_Label"] = np.where(merchant_risk["Prediction"].eq(1), "Flagged", "Normal")

    output_dir.mkdir(parents=True, exist_ok=True)
    captured_tx = merged.loc[merged["Status"].eq("Captured")].copy()
    captured_tx.to_csv(output_dir / "transactions_original_captured.csv.gz", index=False, compression="gzip")
    daily.drop(columns=["Candidate_Extreme_Day"]).rename(
        columns={"High_Confidence_Day": "High_Confidence_Anomaly_Day"}
    ).to_csv(output_dir / "daily_sales_prepared.csv", index=False)
    business_daily.assign(
        Business_Deviation=business_daily["Business_Sales"] - business_daily["Expected_Business_Sales"],
        Abs_Business_Deviation=(business_daily["Business_Sales"] - business_daily["Expected_Business_Sales"]).abs(),
    ).to_csv(output_dir / "business_daily_prepared.csv", index=False)
    merchant_risk.sort_values("Merchant_ID").to_csv(
        output_dir / "merchant_risk.csv", index=False
    )
    save_submission(merchant, merchant_features.index, iso_pred, output_dir / "predictions_original_isolation_forest.csv")
    # Supervised models are trained on original IF pseudo-labels. Original-set
    # predictions are IN-SAMPLE displays only, not independent evaluation.
    ordered_original = merchant.set_index("Merchant_ID").loc[merchant_features.index].reset_index()
    supervised_models = {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
        "Random Forest": RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }
    for label, fitted in supervised_models.items():
        if label == "Gradient Boosting":
            fitted.fit(X, iso_pred, sample_weight=compute_sample_weight(class_weight="balanced", y=iso_pred))
        else:
            fitted.fit(X, iso_pred)
        suffix = label.lower().replace(" ", "_")
        # Original supervised models are trained here; only the five evaluated CSVs are exported.

    return {
        "merchant": merchant,
        "daily": daily,
        "business_daily": business_daily,
        "features": merchant_features,
        "X": X,
        "medians": medians,
        "iso_model": iso_model,
        "iso_pred": iso_pred,
        "merchant_risk": merchant_risk,
        "supervised_models": supervised_models,
    }

## 4. Apply original-fitted models to the New portfolio

Keep original preprocessing, the fitted Isolation Forest and the trained Logistic Regression, Random Forest and Gradient Boosting models unchanged. Produce exactly five Model Evaluator-format CSVs, each containing `Merchant,Prediction` with 105 unique merchants.

In [ ]:
def train_new_if_available(data_dir: Path, output_dir: Path, original_state: dict):
    if not all((data_dir / f).exists() for f in NEW_FILES.values()):
        print("New data files not found. Skipping Deliverable 2 prediction exports.")
        return None

    paths = locate_files(data_dir, NEW_FILES)
    transactions, merchant_new, business_new, status_new = read_source_files(paths)
    _, daily_new, business_daily_new = prepare_daily_data(transactions, merchant_new, business_new, status_new)
    daily_new, business_daily_new = add_time_series_features(daily_new, business_daily_new)
    features_new = build_merchant_features(daily_new)
    X_new, _ = prepare_X(features_new, original_state["medians"])

    X_train = original_state["X"]
    y_train = original_state["iso_pred"]
    iso_model = original_state["iso_model"]

    merchant_new_ordered = merchant_new.set_index("Merchant_ID").loc[features_new.index].reset_index()

    pred_if = (iso_model.predict(X_new) == -1).astype(int)
    save_submission(merchant_new_ordered, features_new.index, pred_if, output_dir / "predictions_new_isolation_forest.csv")

    fitted = original_state["supervised_models"]
    pred_lr = fitted["Logistic Regression"].predict(X_new)
    save_submission(merchant_new_ordered, features_new.index, pred_lr, output_dir / "predictions_new_logistic_regression.csv")
    pred_rf = fitted["Random Forest"].predict(X_new)
    save_submission(merchant_new_ordered, features_new.index, pred_rf, output_dir / "predictions_new_random_forest.csv")
    pred_gb = fitted["Gradient Boosting"].predict(X_new)
    save_submission(merchant_new_ordered, features_new.index, pred_gb, output_dir / "predictions_new_gradient_boosting.csv")

    merged_new, _, _ = prepare_daily_data(transactions, merchant_new, business_new, status_new)
    merged_new.loc[merged_new["Status"].eq("Captured")].to_csv(
        output_dir / "transactions_new_captured.csv.gz", index=False, compression="gzip")
    daily_new.to_csv(output_dir / "daily_sales_new_prepared.csv", index=False)
    business_daily_new.to_csv(output_dir / "business_daily_new_prepared.csv", index=False)
    return {
        "daily_new": daily_new,
        "business_daily_new": business_daily_new,
        "features_new": features_new,
        "X_new": X_new,
        "merchant_new": merchant_new_ordered,
        "predictions": {"isolation_forest": pred_if, "logistic_regression": pred_lr, "random_forest": pred_rf, "gradient_boosting": pred_gb},
    }

In [ ]:
REGENERATED_DIR.mkdir(parents=True,exist_ok=True)
original_state = train_original(RAW_DIR, REGENERATED_DIR)
new_state = train_new_if_available(RAW_DIR, REGENERATED_DIR, original_state)
assert new_state is not None, 'New portfolio source data missing.'
print('Original merchant classifications:', original_state['merchant_risk']['Prediction'].value_counts().to_dict())
for name in ['predictions_original_isolation_forest.csv','predictions_new_isolation_forest.csv','predictions_new_random_forest.csv','predictions_new_gradient_boosting.csv','predictions_new_logistic_regression.csv']:
    pred = pd.read_csv(REGENERATED_DIR/name)
    print(f'{name}: {len(pred)} merchants, {int(pred.Prediction.sum())} anomalous')

## 5. Original vs New — quantitative findings and visual evidence

These plots use the **newly regenerated data**, not hard-coded dashboard images. `Merchant_ID` values from different portfolios must not be merged as the same person. The purpose is to compare portfolio coverage, historical outliers and the ability to drill into category deviations.

In [ ]:
daily_orig = pd.read_csv(REGENERATED_DIR/'daily_sales_prepared.csv',parse_dates=['Date'])
daily_new = pd.read_csv(REGENERATED_DIR/'daily_sales_new_prepared.csv',parse_dates=['Date'])
biz_orig = pd.read_csv(REGENERATED_DIR/'business_daily_prepared.csv',parse_dates=['Date'])
biz_new = pd.read_csv(REGENERATED_DIR/'business_daily_new_prepared.csv',parse_dates=['Date'])
coverage = pd.DataFrame({
    'Portfolio':['Original','New'],
    'Merchants':[daily_orig.Merchant.nunique(),daily_new.Merchant.nunique()],
    'Dates':[daily_orig.Date.nunique(),daily_new.Date.nunique()],
    'Merchant-days':[len(daily_orig),len(daily_new)],
    'Captured transactions':[int(daily_orig.Transaction_Count.sum()),int(daily_new.Transaction_Count.sum())],
    'Captured sales (USD)':[daily_orig.Daily_Sales.sum(),daily_new.Daily_Sales.sum()]
})
display(coverage)
fig,ax=plt.subplots(figsize=(8,3.2))
ax.bar(coverage['Portfolio'],coverage['Captured sales (USD)']/1e6)
ax.set_ylabel('Captured sales (USD millions)');ax.set_title('Two independent synthetic merchant portfolios')
for i,v in enumerate(coverage['Captured sales (USD)']/1e6):ax.text(i,v+1,f'{v:.2f}m',ha='center')
ax.set_ylim(0,175);plt.tight_layout();plt.show()

In [ ]:
sample = daily_orig[daily_orig['Merchant'].eq('Rachel Sheppard')].sort_values('Date')
flag = sample[sample['High_Confidence_Anomaly_Day'].astype(bool)]
fig,ax=plt.subplots(figsize=(11,3.5))
ax.plot(sample.Date,sample.Daily_Sales,label='Captured daily sales',lw=1.4)
ax.plot(sample.Date,sample.Expected_Sales,linestyle='--',label='Prior-only expected sales')
ax.scatter(flag.Date,flag.Daily_Sales,color='red',s=48,zorder=5,label='Flagged day')
ax.set(title='Original: Rachel Sheppard — daily sales anomaly',ylabel='USD');ax.legend(loc='best')
plt.tight_layout();plt.show()
if not flag.empty:
    row=flag.iloc[(flag['Log_Deviation'].abs()).argmax()]
    d=row['Date']; cat=biz_orig[(biz_orig['Merchant']=='Rachel Sheppard') & (biz_orig['Date']==d)].copy()
    cat['Difference_USD']=cat['Business_Sales']-cat['Expected_Business_Sales']
    display(cat[['Business','Business_Sales','Expected_Business_Sales','Difference_USD']].sort_values('Difference_USD'))
    print('Selected historical anomaly:',d.date(), 'actual',round(row.Daily_Sales,2),'expected',round(row.Expected_Sales,2))

## 6. External Model Evaluator — recorded results, not recalculated labels

The university's hidden-label evaluator recorded the following aggregate performance results. Predictions are generated above; **accuracy and confusion-matrix scores cannot be computed from the CSVs alone** because hidden ground-truth labels were not provided.

Across the 105 New merchants, Gradient Boosting predicts **13 anomalies: 12 true positives and 1 false positive**. This is not a conflict with its evaluator F1 score.

In [ ]:
recorded_evaluator = pd.DataFrame([
    ['Original','Isolation Forest',94.3,90.9,66.7,76.9,98.9,82.8,10,1,5,89],
    ['New','Isolation Forest',98.1,100,86.7,92.9,100,93.4,13,0,2,90],
    ['New','Random Forest',98.1,100,86.7,92.9,100,93.4,13,0,2,90],
    ['New','Gradient Boosting',96.2,92.3,80.0,85.7,98.9,89.5,12,1,3,89],
    ['New','Logistic Regression',100,100,100,100,100,100,15,0,0,90]
],columns=['Portfolio','Model','Accuracy %','Precision %','Recall %','F1 %','Specificity %','Balanced accuracy %','TP','FP','FN','TN'])
display(recorded_evaluator)
fig,ax=plt.subplots(figsize=(9,3.4))
new_eval=recorded_evaluator.query('Portfolio == "New"')
ax.barh(new_eval.Model,new_eval['F1 %']);ax.set_xlim(0,110);ax.set_xlabel('Recorded external F1 (%)');ax.invert_yaxis()
for i,v in enumerate(new_eval['F1 %']):ax.text(v+1,i,f'{v:.1f}%')
plt.tight_layout();plt.show()

## 7. Reconciliation against the final GitHub V6 outputs

This is the core release gate. Every prepared file is independently regenerated. The notebook compares its **exact column schema, ordered merchant/day records and numerical values** with the reference. It compares model prediction files at the **raw byte/hash level**, because these determine the published evaluator submissions. It compares transaction GZIP extracts after decompression, because GZIP metadata can change even when all transaction rows are unchanged.

Tiny machine-float serialization differences (around 1e-15) are allowed for prepared numerical features. **Any changed merchant decision, missing row, changed currency amount, or changed table schema causes the final release gate to fail.** The Original feature naming and Business_Deviation conventions are preserved exactly.

In [ ]:
import hashlib
PREDICTION_FILES = [
 'predictions_original_isolation_forest.csv','predictions_new_isolation_forest.csv',
 'predictions_new_random_forest.csv','predictions_new_gradient_boosting.csv',
 'predictions_new_logistic_regression.csv'
]
GITHUB_FILES = [
 'daily_sales_prepared.csv','business_daily_prepared.csv','merchant_risk.csv',
 'daily_sales_new_prepared.csv','business_daily_new_prepared.csv',
 *PREDICTION_FILES,
 'transactions_original_captured.csv.gz','transactions_new_captured.csv.gz'
]
def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as stream:
  for block in iter(lambda:stream.read(1<<20),b''):h.update(block)
 return h.hexdigest()

def compare_one(name):
 ref = REFERENCE_DIR/name; gen = REGENERATED_DIR/name
 if not (ref.exists() and gen.exists()):return dict(File=name,Rows=0,Schema='MISSING',Match=False,Byte_identical=False,Details='Missing output')
 byte = sha256(ref)==sha256(gen)
 if name in PREDICTION_FILES:
  a=pd.read_csv(ref);b=pd.read_csv(gen)
  ok=byte and len(a)==105 and a.Merchant.nunique()==105 and list(a.columns)==['Merchant','Prediction']
  return dict(File=name,Rows=len(a),Schema='OK' if list(a)==list(b) else 'DIFF',Match=ok,Byte_identical=byte,Details='Exact prediction bytes' if ok else 'PREDICTION MISMATCH')
 a=pd.read_csv(ref,low_memory=False);b=pd.read_csv(gen,low_memory=False)
 if list(a.columns)!=list(b.columns) or a.shape!=b.shape:
  return dict(File=name,Rows=len(a),Schema='DIFF',Match=False,Byte_identical=byte,Details=f'{a.shape} vs {b.shape}')
 if name=='merchant_risk.csv':
  a=a.sort_values('Merchant_ID').reset_index(drop=True)
  b=b.sort_values('Merchant_ID').reset_index(drop=True)
 elif name.endswith('.csv.gz'):
  # Same transaction sequence after decompression, including status and amounts.
  pass
 try:
  pd.testing.assert_frame_equal(a,b,check_dtype=False,check_exact=False,rtol=1e-11,atol=1e-8)
  ok=True;reason='Analytical values and ordered records match'
 except AssertionError as exc:
  ok=False;reason=str(exc).splitlines()[0][:110]
 return dict(File=name,Rows=len(a),Schema='OK',Match=ok,Byte_identical=byte,Details=reason)

reconciliation = pd.DataFrame([compare_one(n) for n in GITHUB_FILES])
display(reconciliation)
assert reconciliation['Match'].all(), 'STOP: at least one output does not match GitHub V6.'
assert reconciliation[reconciliation.File.isin(PREDICTION_FILES)]['Byte_identical'].all(), 'Evaluator prediction bytes changed.'
print('RECONCILIATION: PASSED — all 12 V6 output files agree with regenerated analytical results.')

## 8. Final verified outputs and Colab downloads

Only after all reconciliation tests pass does this cell copy the **published reference files byte-for-byte** into a verified release folder. This copy is intentional: it preserves the GitHub V6 release (including existing CSV precision and GZIP encoding) rather than asserting byte-identical regeneration for floating-point prepared files.

The notebook also exports `reconciliation_report.csv` documenting which outputs were byte-identical and which were validated numerically. **Do not replace GitHub outputs with an unverified model rerun.**

In [ ]:
import zipfile,shutil
VERIFIED_DIR.mkdir(parents=True,exist_ok=True)
for name in GITHUB_FILES:
 shutil.copy2(REFERENCE_DIR/name, VERIFIED_DIR/name)
assert all(sha256(REFERENCE_DIR/name)==sha256(VERIFIED_DIR/name) for name in GITHUB_FILES)
report_path=WORKDIR/'reconciliation_report.csv'
reconciliation.to_csv(report_path,index=False)
zip_path=WORKDIR/'Merchant_Verified_GitHub_V6_Outputs.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for name in GITHUB_FILES:z.write(VERIFIED_DIR/name,'outputs/'+name)
 z.write(report_path,'reconciliation_report.csv')
print('Verified release ZIP:',zip_path,'size (MB):',round(zip_path.stat().st_size/1e6,2))
print('Five evaluator-ready CSV files are preserved exactly.')
if 'google.colab' in sys.modules:
 from google.colab import files
 files.download(str(zip_path))

## 9. Assignment completion and interpretation

**Deliverable 1:** source understanding, captured-only daily series, unsupervised Isolation Forest, original merchant list and external metric record. **Deliverable 2:** three supervised alternatives trained on Original pseudo-labels, New-merchant classifications, original-fitted Isolation Forest on New data, and comparative external metric record. 

**Scope:** Two synthetic 105-merchant portfolios covering the same 90 calendar days. New = different merchants, *not* an out-of-time future holdout. The original classifier's 66.7% external recall demonstrates missed evaluator anomalies; perfect New Logistic Regression scores do not establish real-bank reliability.

**Code/output provenance:** The final V6 GitHub dashboard remains a separate application. This notebook builds **no app**. The source archive is private to Colab, the reference files represent the published GitHub outputs, and the separate reconciliation report records the analytical reproduction result.

**Literature (post-2018):** Blázquez-García et al. (2021), *ACM Computing Surveys* (time-series anomaly detection); Pang et al. (2021), *ACM Computing Surveys* (anomaly learning methods); Carcillo et al. (2021), *Information Sciences* (supervised/unsupervised combination); Studer et al. (2021), *Machine Learning and Knowledge Extraction* (CRISP-ML(Q)); Hilal et al. (2022), *Expert Systems with Applications* (financial anomaly detection). See the final written report for bibliographic details.